## **Notebook 11 — LangGraph RAG Pipeline**
- Wire the full RAG pipeline into a LangGraph StateGraph. Each retrieval and generation step becomes a node. Edges define the flow. A conditional edge adds a retry loop — if the answer has low confidence, the graph automatically retries with a broader search before giving up.

- Document used: `NLTK book` corpus (natural language processing textbook).
Why LangGraph over a plain function: A plain function runs linearly — no branching, no retries, no inspection. LangGraph gives you a visible graph with named nodes, typed state at every step, conditional routing, and built-in tracing. When something fails in production you can see exactly which node failed and what state it had.

## ***Part 1: Setup & Imports***

- LangGraph is a library built on top of LangChain that lets you define pipelines as directed graphs. StateGraph is the main class — you define a typed state, add nodes (functions), connect them with edges, and compile into a runnable chain.

- TypedDict for state: LangGraph passes one state object between every node. Using TypedDict makes every field explicit and type-checked. When you move to .py files you will replace this with a Pydantic model — the structure stays identical.

- Annotated + operator: Some fields use Annotated[list, operator.add]. This tells LangGraph how to merge state when nodes run in parallel — instead of overwriting the list, it appends to it. Used for the retrieved_chunks field since multiple retrieval nodes could contribute results

In [2]:
import os, json, re, operator
from pathlib import Path
from typing import TypedDict, Annotated, Optional
from dataclasses import dataclass, field
from collections import Counter
from dotenv import load_dotenv

import numpy as np
import psycopg2
from pgvector.psycopg2 import register_vector
import bm25s

from FlagEmbedding import BGEM3FlagModel, FlagReranker
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# LangGraph core imports
from langgraph.graph import StateGraph, END

load_dotenv()

DATABASE_SYNC = os.getenv("DATABASE_URL_SYNC")
INDEX_DIR     = Path("../data/bm25_index_nltk")
INDEX_DIR.mkdir(parents=True, exist_ok=True)

print("All imports OK")
print(f"LangGraph imported successfully")

All imports OK
LangGraph imported successfully


## **Cell 2 — define RAG state**
- The state is the most important design decision in LangGraph. Every node receives the full state as input and returns a partial state update as output. LangGraph merges the update into the current state before calling the next node.

**Field by field:**
- query: the original user question. Never modified after it is set.
- hyde_doc: the hypothetical answer generated by HyDE. Set by the transform node.
- hyde_embedding: the vector of the HyDE doc. Set by the transform node, used by dense search.
- retrieved_chunks: uses Annotated[list, operator.add] so parallel nodes append rather than overwrite.
- fused_chunks: RRF output — merged and re-ranked candidates.
- reranked_chunks: final top-5 after cross-encoder scoring.
- answer: the LLM-generated response.
- confidence: average reranker score of top-3 chunks. Used by the conditional edge to decide retry.
- retry_count: how many retries have happened. Prevents infinite loops.
- sources: citation metadata returned to the user.
- error: if any node fails, it writes here and the graph routes to END gracefully.

In [3]:
class RAGState(TypedDict):
    """
    The single state object that flows through every node in the graph.
    Every node reads from this and writes partial updates back to it.
    LangGraph merges updates automatically between nodes.
    """
    # Input
    query           : str

    # Query transformation (HyDE node)
    hyde_doc        : Optional[str]
    hyde_embedding  : Optional[list[float]]

    # Retrieval nodes — Annotated[list, operator.add] means
    # if two nodes both return retrieved_chunks, the lists are
    # concatenated instead of one overwriting the other
    retrieved_chunks: Annotated[list, operator.add]

    # Fusion + reranking
    fused_chunks    : list
    reranked_chunks : list

    # Generation
    answer          : Optional[str]
    sources         : list

    # Control flow
    confidence      : float    # avg rerank score of top-3, used for retry decision
    retry_count     : int      # incremented on each retry, max=2
    error           : Optional[str]

print("RAGState defined")
print("Fields:", list(RAGState.__annotations__.keys()))

RAGState defined
Fields: ['query', 'hyde_doc', 'hyde_embedding', 'retrieved_chunks', 'fused_chunks', 'reranked_chunks', 'answer', 'sources', 'confidence', 'retry_count', 'error']
